# CPMK3 NRP YOLO11s

Run tambahan melatih ukuran 320 dan 960 dengan split, seed, dan recipe model 640. Perbandingan ukuran menggunakan validation. Data berasal dari ekspor COCO v2.

Dicatat 23 November 2026 20:38:48 WIB


## 1. Dependensi dan import

Instal hanya pustaka tambahan. PyTorch/torchvision bawaan Kaggle dipakai; uji NMS mendeteksi konflik lebih awal. Internet diperlukan saat pustaka/bobot belum tersedia. Versi Ultralytics dipatok agar API tidak bergeser.

In [ ]:
# Dependensi tambahan; PyTorch/CUDA bawaan Kaggle tidak diganti.
import os, sys, subprocess, importlib.util, importlib.metadata
os.environ.setdefault('YOLO_CONFIG_DIR', '/kaggle/working/yolo-settings' if os.path.isdir('/kaggle/working') else '/tmp/cpmk3-yolo-settings')
os.makedirs(os.environ['YOLO_CONFIG_DIR'], exist_ok=True)
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cpmk3-matplotlib')
required = {'ultralytics': 'ultralytics==8.3.228', 'pycocotools': 'pycocotools==2.0.11',
            'yaml': 'pyyaml', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
            'PIL': 'pillow', 'cv2': 'opencv-python-headless', 'scipy': 'scipy', 'IPython': 'ipython'}
install = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if importlib.util.find_spec('ultralytics') is not None and importlib.metadata.version('ultralytics') != '8.3.228':
    install.append('ultralytics==8.3.228')
if install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', *install])
import copy, csv, gc, hashlib, io, json, math, random, re, shutil, time, warnings, zipfile
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from datetime import datetime, timezone
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw, ImageFont, ImageOps, ExifTags
import torch, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops.misc import FrozenBatchNorm2d
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
from IPython.display import display
# Tangkap konflik torch/torchvision sebelum berjam-jam training.
_ = torchvision.ops.nms(torch.tensor([[0., 0., 2., 2.]]), torch.tensor([1.]), 0.5)
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| ultralytics', importlib.metadata.version('ultralytics'))


## 2. Konfigurasi

Awalnya hanya mode audit. NRP 0 telah dikonfirmasi. `DATASET_OVERRIDE=None` adalah pilihan utama; isi path hanya jika ada beberapa ekspor berbeda. `HP_LABELS_REVIEWED` adalah deklarasi nyata, bukan tombol mengabaikan masalah label.

In [ ]:
# ===== BAGIAN YANG BOLEH KAMU UBAH =====
INPUT_ROOT = Path('/kaggle/input')
# Biarkan None untuk deteksi otomatis. Bila ambigu, isi folder ekspor COCO atau ZIP v2 yang tepat.
DATASET_OVERRIDE = None
WORK_ROOT = Path('/kaggle/working/cpmk3_final')
NRP_LAST_DIGIT = 0
RUN_TRAINING = True             # True setelah provenance data latih dikonfirmasi
RUN_EVALUATION = True            # Otomatis menunggu checkpoint jika training belum dijalankan
RUN_NRP_EXPERIMENT = True        # NRP 0: latih YOLO11n pada 320, 640, 960
RUN_FAILURE_ANALYSIS = False
SOURCE_V5_CONFIRMED = False      # True HANYA jika ada bukti asal data latih dari v5 / persetujuan dosen
SOURCE_V5_NOTE = 'Porsi publik Roboflow v2 cocok hitungan v5 (397/166/359); kesamaan tiap gambar dan label belum dibuktikan. Pelatihan ini memakai ekspor v2.'              # Tulis URL/bukti asal atau keputusan dosen yang sebenarnya
HP_LABELS_REVIEWED = False       # True setelah review kelengkapan/ketepatan semua mask HP
# Optional: folder input yang berisi file foto asli dengan EXIF dan screenshot proses anotasi.
ORIGINAL_PHOTOS_DIR = None
SCREENSHOTS_DIR = None
# ===== Pengaturan tetap percobaan =====
SEED = 42
BASE_SIZE = 640
RESOLUTIONS = (320, 640, 960)
EPOCHS = 200                     # Batas epoch; deadline waktu bisa menghentikan lebih awal
BATCH_YOLO = 4                   # Sama untuk ketiga ukuran NRP; turunkan SEMUA varian jika OOM
BATCH_MASK = 1
WORKERS = 0                     # Aman untuk notebook; dapat dinaikkan menjadi 2 bila perlu
YOLO_N_MINUTES = 20             # 640: baseline sekaligus varian NRP
YOLO_S_MINUTES = 30
MASK_MINUTES = 40
NRP_MINUTES = 20                # 320 dan 960; total YOLO11n sekitar 60 menit
MAX_SECONDS_PER_MODEL = 7200
PRED_CONF = 0.001               # Ambang rendah untuk kurva precision-recall AP
PRED_MAX_DET = 100
FAIL_SCORE_THR = 0.50
MASK_LR = 0.0025
# Gambar dengan dimensi piksel != metadata COCO dikarantina, bukan diputar/diregangkan dengan tebakan.
QUARANTINE_DIMENSION_MISMATCH = True
# Asal 40 foto HP dari ZIP Foto_40_Terpilih_JPG_Roboflow.zip, BUKAN regex tanggal.
HP_SOURCE_NAMES = set('''20260713_111506.jpg 20260714_105357.jpg 20260715_125526.jpg
20260804_120415.jpg 20260805_114312.jpg 20260812_110307.jpg 20260826_154026.jpg
20260903_073220.jpg 20260903_161001.jpg 20260903_161212.jpg 20261030_143151.jpg
20261118_083036.jpg 20261118_083047.jpg 20261118_083050.jpg 20261118_084035.jpg
20261118_084038.jpg 20261118_084046.jpg 20261118_084140.jpg 20261118_084148.jpg
20261118_084152.jpg 20261118_134539.jpg 20261118_134543.jpg 20261118_134549.jpg
20261118_192611.jpg 20261118_192615.jpg 20261118_192621.jpg 20261118_192648.jpg
20261118_192856.jpg 20261118_192900.jpg 20261118_192904.jpg 20261118_192913.jpg
20261118_192916.jpg 20261118_192932.jpg 20261118_192936.jpg 20261118_192940.jpg
20261118_192944.jpg 20261118_193001.jpg 20261118_193023.jpg 20261118_193029.jpg
20261118_193033.jpg'''.split())
assert len(HP_SOURCE_NAMES) == 40
assert NRP_LAST_DIGIT == 0, 'Notebook ini disesuaikan untuk digit NRP 0 yang kamu konfirmasi.'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
seed_all()
torch.backends.cudnn.benchmark = False
def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    tmp.replace(path)
def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()
def source_name(info):
    original = info.get('extra', {}).get('name')
    if original: return Path(original).name
    name = Path(info['file_name']).name
    # Roboflow: source_jpg.rf.HASH.jpg -> source.jpg
    return re.sub(r'_(jpg|jpeg|png)\.rf\.[^.]+\.[^.]+$', lambda m: '.' + m.group(1), name, flags=re.I)
def is_hp(info): return source_name(info) in HP_SOURCE_NAMES
print('Device:', DEVICE, '| input:', INPUT_ROOT, '| output:', WORK_ROOT)


## 3. Deteksi input dan ekstraksi aman

Cari struktur train/valid/test dan COCO JSON, abaikan __MACOSX/._, tolak traversal/symlink. Kandidat v1 ditolak karena identitas HP berada di train. ZIP foto dan folder audit tidak dianggap dataset training.

In [ ]:
def ignored(path):
    return any(part == '__MACOSX' or part.startswith('.') for part in Path(path).parts)
def safe_extract(archive, dest):
    dest = Path(dest).resolve()
    with zipfile.ZipFile(archive) as z:
        infos = [x for x in z.infolist() if not ignored(x.filename)]
        if sum(x.file_size for x in infos) > 8_000_000_000:
            raise ValueError('Arsip melebihi batas ekstraksi 8 GB.')
        for item in z.infolist():
            target = (dest / item.filename).resolve()
            if not target.is_relative_to(dest) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Path/symlink tidak aman di ZIP: ' + item.filename)
        for item in infos: z.extract(item, dest)
    return dest

def split_paths(root):
    paths = {}
    for sp, aliases in {'train': ('train',), 'valid': ('valid', 'val', 'validation'), 'test': ('test',)}.items():
        found = [root / a / '_annotations.coco.json' for a in aliases if (root / a / '_annotations.coco.json').is_file()]
        if len(found) != 1: return None
        paths[sp] = found[0]
    return paths

def inspect_candidate(paths):
    data = {sp: json.loads(p.read_text()) for sp, p in paths.items()}
    hp = {sp: sum(is_hp(i) for i in d['images']) for sp, d in data.items()}
    counts = {sp: len(d['images']) for sp, d in data.items()}
    return {'paths': paths, 'data': data, 'counts': counts, 'hp_counts': hp,
            'eligible': hp == {'train': 0, 'valid': 0, 'test': 40}}

# Menampilkan input yang benar-benar dipasang, bukan hardcode slug Kaggle.
if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(f'{INPUT_ROOT} tidak ada. Di Kaggle pilih Add Input; untuk uji lokal ubah INPUT_ROOT.')
for path in sorted(INPUT_ROOT.iterdir()):
    n_images = sum(1 for p in path.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')) if path.is_dir() else 0
    print(path, '| gambar terurai:', n_images)
search_root = Path(DATASET_OVERRIDE) if DATASET_OVERRIDE else INPUT_ROOT
if not search_root.exists(): raise FileNotFoundError(search_root)
candidates = []
if search_root.is_dir():
    roots = sorted({p.parent.parent for p in search_root.rglob('_annotations.coco.json') if not ignored(p)})
    for root in roots:
        paths = split_paths(root)
        if paths: candidates.append(inspect_candidate(paths))
archives = [search_root] if search_root.is_file() else sorted(search_root.rglob('*.zip'))
for archive in archives:
    if ignored(archive): continue
    with zipfile.ZipFile(archive) as z:
        names = [n for n in z.namelist() if n.endswith('/_annotations.coco.json') and not ignored(n)]
        prefixes = {str(Path(n).parent.parent) for n in names}
        for prefix in prefixes:
            def zn(sp): return str(Path(prefix) / sp / '_annotations.coco.json')
            if not all(zn(sp) in names for sp in ('train', 'valid', 'test')): continue
            meta = {sp: json.loads(z.read(zn(sp))) for sp in ('train', 'valid', 'test')}
            hpc = {sp: sum(is_hp(i) for i in d['images']) for sp, d in meta.items()}
            print('ZIP:', archive.name, '| jumlah:', {sp:len(d['images']) for sp,d in meta.items()}, '| HP:', hpc)
            # V1 memiliki HP di train: ditolak sebelum ekstraksi.
            if hpc != {'train': 0, 'valid': 0, 'test': 40}: continue
            extracted = WORK_ROOT / 'extracted' / sha_file(archive)[:16]
            safe_extract(archive, extracted)
            paths = split_paths(extracted / prefix)
            if paths: candidates.append(inspect_candidate(paths))
valid_candidates = [c for c in candidates if c['eligible']]
# Salinan dataset yang identik (mis. ZIP + hasil ekstraknya) tidak membuat pilihan ambigu.
def signature(c):
    return hashlib.sha256(json.dumps(c['data'], sort_keys=True).encode()).hexdigest()
unique = {signature(c): c for c in valid_candidates}
if len(unique) != 1:
    raise RuntimeError(f'Ditemukan {len(unique)} dataset valid. Pasang ZIP v2 final-test40, bukan v1. '
                       'Jika ada beberapa ekspor berbeda, isi DATASET_OVERRIDE dengan satu path tepat.')
CHOSEN = next(iter(unique.values()))
RAW = CHOSEN['data']
RAW_PATHS = CHOSEN['paths']
RAW_DIRS = {sp: p.parent for sp,p in RAW_PATHS.items()}
print('Dipilih:', RAW_PATHS, '\nJumlah sebelum audit:', CHOSEN['counts'])
print('V1, ZIP foto tanpa label, dan folder preview audit tidak dipakai sebagai training tambahan.')


## 4. Audit gambar, leakage, kelas, dan geometri

Periksa piksel vs ukuran JSON, identitas sumber, hash piksel lintas split, referensi ID dan decoding seluruh mask. Audit ini tidak menangkap semua near-duplicate atau kebocoran dari data pretraining eksternal. Empat mismatch dikarantina; tidak ada resplit acak. Simpan sumber tetap utuh; pada salinan canonical area dan bbox dihitung dari mask asli agar COCO size metrics tidak memakai area kotak. Polygon/RLE, lubang, dan komponen terpisah dipertahankan.

In [ ]:
def encode_mask(mask):
    r = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return {'size': [int(x) for x in r['size']], 'counts': r['counts'].decode('ascii')}
def decode_seg(seg, h, w):
    if isinstance(seg, list):
        polygons = [p for p in seg if len(p) >= 6 and len(p) % 2 == 0]
        if len(polygons) != len(seg) or not polygons: raise ValueError('Polygon kosong/tidak valid')
        r = mask_utils.merge(mask_utils.frPyObjects(polygons, h, w))
    elif isinstance(seg, dict):
        if list(seg['size']) != [h, w]: raise ValueError('Ukuran RLE tidak cocok')
        r = mask_utils.frPyObjects(seg, h, w) if isinstance(seg['counts'], list) else {
            'size': seg['size'], 'counts': seg['counts'].encode('ascii') if isinstance(seg['counts'], str) else seg['counts']}
    else: raise ValueError('Format segmentation tidak dikenali')
    mask = mask_utils.decode(r)
    if mask.ndim == 3: mask = np.any(mask, axis=2)
    return mask.astype(np.uint8)
def make_coco(data):
    c = COCO(); c.dataset = copy.deepcopy(data)
    with redirect_stdout(io.StringIO()): c.createIndex()
    return c

# Fingerprint semua anotasi dan seluruh file gambar: mencegah cache checkpoint dari dataset lain.
hasher = hashlib.sha256(b'cpmk3-canonical-v3-maskarea-quarantine')
file_records, dimension_bad, corrupt, hash_groups, source_groups = [], [], [], defaultdict(list), defaultdict(list)
for sp in ('train', 'valid', 'test'):
    data = RAW[sp]
    assert len({i['id'] for i in data['images']}) == len(data['images']), f'Duplicate image ID: {sp}'
    assert len({a['id'] for a in data['annotations']}) == len(data['annotations']), f'Duplicate annotation ID: {sp}'
    hasher.update(json.dumps(data, sort_keys=True).encode())
    for info in data['images']:
        f = (RAW_DIRS[sp] / info['file_name']).resolve()
        if not f.is_relative_to(RAW_DIRS[sp].resolve()): raise ValueError('Path gambar keluar folder')
        if not f.is_file(): raise FileNotFoundError(f)
        try:
            with Image.open(f) as pic:
                pic.load(); real = pic.size
                pixel_hash = hashlib.sha256(pic.convert('RGB').tobytes()).hexdigest()
        except Exception as exc:
            corrupt.append({'split':sp,'file':info['file_name'],'error':str(exc)}); continue
        digest = sha_file(f)
        record = {'split':sp,'image_id':info['id'],'file':info['file_name'],'source':source_name(info),
                  'sha256':digest,'pixel_sha256':pixel_hash,'width':real[0],'height':real[1],
                  'json_width':info['width'],'json_height':info['height'],'is_hp':is_hp(info)}
        hasher.update(digest.encode()); file_records.append(record)
        hash_groups[pixel_hash].append(record); source_groups[source_name(info)].append(record)
        if real != (info['width'], info['height']): dimension_bad.append(record)
if corrupt: raise RuntimeError(f'Gambar rusak: {corrupt[:5]}')
# Identitas sumber dan piksel sama lintas split -> hentikan, tidak membagi ulang secara acak.
leaks = [g for g in list(hash_groups.values()) + list(source_groups.values()) if len({r['split'] for r in g}) > 1]
if leaks: raise RuntimeError(f'Data leakage lintas split: {leaks[:3]}')
DATA_FINGERPRINT = hasher.hexdigest()
WORK = WORK_ROOT / DATA_FINGERPRINT[:16]
PROOF, METRICS, RUNS = WORK/'bukti', WORK/'metrik', WORK/'runs'
for folder in (PROOF,METRICS,RUNS): folder.mkdir(parents=True,exist_ok=True)
pd.DataFrame(file_records).to_csv(PROOF/'inventaris_gambar.csv',index=False)
pd.DataFrame(dimension_bad).to_csv(PROOF/'karantina_dimensi.csv',index=False)
write_json(PROOF/'duplicate_sources_within_split.json', [g for g in source_groups.values() if len(g)>1])
if dimension_bad and not QUARANTINE_DIMENSION_MISMATCH:
    raise ValueError('Perbaiki orientasi gambar/mask di Roboflow; jangan menebak resize otomatis.')
quarantined = {(r['split'],r['image_id']) for r in dimension_bad}
if any(sp == 'test' for sp,_ in quarantined):
    raise ValueError('Ada dimensi test salah. Perbaiki ekspor dahulu agar 40 HP/359 asli tetap utuh.')

# Konsistensi kelas diperiksa berdasarkan ID DAN nama pada seluruh split.
CAT_NAMES = {int(c['id']):c['name'] for c in RAW['train']['categories']}
assert len(set(CAT_NAMES.values())) == len(CAT_NAMES), 'Nama kategori duplikat'
for sp,d in RAW.items():
    assert {int(c['id']):c['name'] for c in d['categories']} == CAT_NAMES, f'Pemetaan kelas berbeda pada {sp}'
used_raw = {int(a['category_id']) for d in RAW.values() for a in d['annotations']}
PLACEHOLDER_IDS = {c['id'] for c in RAW['train']['categories']
                   if c['name']=='indoor-objects' and c.get('supercategory')=='none' and c['id'] not in used_raw}
CAT_IDS = sorted(set(CAT_NAMES)-PLACEHOLDER_IDS)
CAT2INDEX = {cid:k for k,cid in enumerate(CAT_IDS)}
INDEX2CAT = {k:cid for cid,k in CAT2INDEX.items()}
NC = len(CAT_IDS)
CATEGORIES = [copy.deepcopy(c) for c in RAW['train']['categories'] if c['id'] in CAT_IDS]
COCO_DATA, geometry_report, decode_errors = {}, [], []
for sp,d in RAW.items():
    images = [copy.deepcopy(i) for i in d['images'] if (sp,i['id']) not in quarantined]
    lookup = {i['id']:i for i in images}
    annotations = []
    for ann in d['annotations']:
        if ann['image_id'] not in {i['id'] for i in d['images']}: raise ValueError('Annotation image_id tidak ada')
        if ann['image_id'] not in lookup: continue
        info = lookup[ann['image_id']]
        try:
            if ann['category_id'] not in CAT2INDEX: raise ValueError('category_id tidak dikenal')
            m = decode_seg(ann.get('segmentation'), info['height'], info['width'])
            if not m.any(): raise ValueError('Mask kosong')
            rle = encode_mask(m)
            rb = {'size':rle['size'],'counts':rle['counts'].encode('ascii')}
            area = int(m.sum()); box = mask_utils.toBbox(rb).astype(float).tolist()
            a = copy.deepcopy(ann)
            # COCO export memberi area = luas bbox pada banyak mask. Hitung geometri yang benar dari mask.
            a.update(segmentation=rle, area=area, bbox=box, iscrowd=int(a.get('iscrowd',0)))
            annotations.append(a)
            geometry_report.append({'split':sp,'annotation_id':a['id'],
                                    'source_area':float(ann.get('area',0)),'mask_area':area,
                                    'area_changed':abs(float(ann.get('area',0))-area)>1})
        except Exception as exc: decode_errors.append({'split':sp,'annotation_id':ann['id'],'error':str(exc)})
    COCO_DATA[sp] = {'info':{'description':'COCO canonical: visible geometry preserved; mask area/bbox recomputed'},
                     'images':images,'annotations':annotations,'categories':CATEGORIES}
if decode_errors:
    write_json(PROOF/'mask_errors.json',decode_errors)
    raise ValueError(f'{len(decode_errors)} mask tidak dapat dibaca; perbaiki sebelum training.')
COCO_GT = {sp:make_coco(d) for sp,d in COCO_DATA.items()}
HP_IMAGES = [i for i in COCO_DATA['test']['images'] if is_hp(i)]
HP_IDS = {i['id'] for i in HP_IMAGES}
OFFICIAL_IDS = {i['id'] for i in COCO_DATA['test']['images'] if not is_hp(i)}
assert len(HP_IDS)==40 and len(OFFICIAL_IDS)==359, 'Identitas test tidak sesuai ekspor yang diaudit.'
assert {source_name(i) for i in HP_IMAGES} == HP_SOURCE_NAMES
assert not any(is_hp(i) for sp in ('train','valid') for i in COCO_DATA[sp]['images'])
train_freq = Counter(a['category_id'] for a in COCO_DATA['train']['annotations'])
valid_freq = Counter(a['category_id'] for a in COCO_DATA['valid']['annotations'])
hp_freq = Counter(a['category_id'] for a in COCO_DATA['test']['annotations'] if a['image_id'] in HP_IDS)
orig_freq = Counter(a['category_id'] for a in COCO_DATA['test']['annotations'] if a['image_id'] in OFFICIAL_IDS)
SEEN_CAT_IDS = sorted(train_freq)
COMMON_CAT_IDS = sorted(set(train_freq)&set(hp_freq)&set(orig_freq))
SCOPES = {'semua_kelas':CAT_IDS,'kelas_ada_di_train':SEEN_CAT_IDS,'kelas_sama_kedua_test':COMMON_CAT_IDS}
for sp,d in COCO_DATA.items(): write_json(PROOF/f'canonical_{sp}.json',d)
write_json(PROOF/'mapping_kelas.json', {'num_classes':NC,'placeholder_removed':sorted(PLACEHOLDER_IDS),
    'yolo_to_coco':INDEX2CAT,'maskrcnn_to_coco':{k+1:v for k,v in INDEX2CAT.items()}})
pd.DataFrame(geometry_report).to_csv(PROOF/'audit_geometri_mask.csv',index=False)
class_table = pd.DataFrame([{'category_id':c,'class':CAT_NAMES[c],'train':train_freq[c],
    'valid':valid_freq[c],'test_original':orig_freq[c],'test_hp':hp_freq[c]} for c in CAT_IDS])
class_table.to_csv(PROOF/'jumlah_instance_per_kelas.csv',index=False)
summary = {'fingerprint':DATA_FINGERPRINT,'raw_counts':CHOSEN['counts'],
    'effective_counts':{sp:len(d['images']) for sp,d in COCO_DATA.items()},
    'annotations':{sp:len(d['annotations']) for sp,d in COCO_DATA.items()},
    'original_test':len(OFFICIAL_IDS),'hp_test':len(HP_IDS),'classes':NC,
    'train_supported_classes':len(SEEN_CAT_IDS),'common_classes':len(COMMON_CAT_IDS),
    'hp_untrained_annotations':sum(v for c,v in hp_freq.items() if not train_freq[c]),
    'hp_untrained_classes':[CAT_NAMES[c] for c in hp_freq if not train_freq[c]],
    'quarantine_count':len(quarantined),'area_recomputed':sum(r['area_changed'] for r in geometry_report),
    'provenance_confirmed':SOURCE_V5_CONFIRMED,'provenance_note':SOURCE_V5_NOTE}
write_json(PROOF/'audit_summary.json',summary)
display(pd.DataFrame([{'split':sp,'images':len(d['images']),'instances':len(d['annotations'])} for sp,d in COCO_DATA.items()]))
print('Test asli:',len(OFFICIAL_IDS),'| HP:',len(HP_IDS),'| kelas:',NC,'| kelas ada di train:',len(SEEN_CAT_IDS))
print('Karantina dimensi:',len(quarantined),'| HP berkelas tanpa contoh train:',summary['hp_untrained_annotations'])
print('Status v5:', 'TERKONFIRMASI' if SOURCE_V5_CONFIRMED else 'BELUM TERKONFIRMASI - tidak boleh diklaim sebagai v5')


## 5. Overlay dan review 40 HP

Isi `review_manual_40_hp.csv` berdasarkan pengamatan. `reviewed_all_objects=YA` hanya setelah seluruh objek dari kelas acuan dicek. Pada foto ruang kelas, cek kursi, orang, meja, layar dan area kelas acuan yang belum bermask. Pada foto kasur, cek tempat tidur dan setiap bantal; satu mask bantal tidak melengkapi seluruh foto. Kelas tak ada di train ditandai; jangan menamai ulang `bagpack` menjadi `backpack` atau `screen` menjadi `tv` tanpa memeriksa definisi sumber. Untuk proyek panoptic yang diekspor sebagai instance, kelas area seperti wall/floor juga mengikuti konvensi anotasi sumber. Overlay yang dihasilkan adalah alat review, bukan screenshot bukti bekerja di Roboflow.

In [ ]:
def load_rgb(split, info):
    with Image.open(RAW_DIRS[split]/info['file_name']) as pic:
        arr = np.asarray(pic.convert('RGB')).copy()
    if arr.shape[:2] != (info['height'],info['width']): raise ValueError('Dimensi gambar berubah sejak audit.')
    return arr



## 6. Konversi Train/Valid ke YOLO

Satu baris per instance dan satu class ID map yang sama. Data test tidak masuk direktori YOLO training. Pada file yang diberikan seluruh train/valid berupa polygon satu komponen, sehingga tidak ada komponen yang perlu dibuang. RLE/multipart baru tidak dikonversi diam-diam dengan mengambil area terbesar: cell menghentikan konversi lossy. COCO asli/canonical tetap dipakai untuk evaluasi kedua keluarga.

In [ ]:
# Train/Valid mempertahankan polygon asal jika ada. Evaluasi selalu memakai mask COCO canonical.
YOLO_DIR = WORK/'yolo_data'
YOLO_YAML = YOLO_DIR/'data.yaml'
conversion = []
for sp,ysp in [('train','train'),('valid','val')]:
    imdir,lbdir = YOLO_DIR/'images'/ysp,YOLO_DIR/'labels'/ysp
    imdir.mkdir(parents=True,exist_ok=True);lbdir.mkdir(parents=True,exist_ok=True)
    raw_ann = {a['id']:a for a in RAW[sp]['annotations']}
    for info in COCO_DATA[sp]['images']:
        dest = imdir/info['file_name']
        if not dest.exists(): shutil.copy2(RAW_DIRS[sp]/info['file_name'],dest)
        lines = []
        for ann in COCO_GT[sp].imgToAnns[info['id']]:
            src = raw_ann[ann['id']]; seg = src.get('segmentation')
            if ann.get('iscrowd',0):
                raise ValueError('YOLO polygon tidak mendukung crowd secara setara; putuskan kebijakan crowd dahulu.')
            if isinstance(seg,list) and len(seg)==1:
                pts = np.asarray(seg[0],dtype=np.float64).reshape(-1,2)
                method = 'original_polygon'
            else:
                # Jangan diam-diam mengambil kontur terbesar/mengisi lubang.
                mask = decode_seg(ann['segmentation'],info['height'],info['width'])
                contours,hierarchy = cv2.findContours(mask,cv2.RETR_TREE,cv2.CHAIN_APPROX_SIMPLE)
                if len(contours)!=1 or len(contours[0])<3:
                    raise ValueError(f"Konversi lossy untuk {sp}/{info['file_name']} ann {ann['id']}; perlu format training mask-preserving.")
                pts = contours[0].reshape(-1,2).astype(float);method='single_contour_from_mask'
            if len(pts)<3 or not np.isfinite(pts).all():raise ValueError('Titik polygon tidak valid')
            pts[:,0] = np.clip(pts[:,0]/info['width'],0,1)
            pts[:,1] = np.clip(pts[:,1]/info['height'],0,1)
            lines.append(' '.join([str(CAT2INDEX[ann['category_id']])]+[f'{v:.8f}' for v in pts.ravel()]))
            conversion.append({'split':sp,'annotation_id':ann['id'],'method':method,'components_dropped':0})
        (lbdir/(Path(info['file_name']).stem+'.txt')).write_text('\n'.join(lines)+'\n' if lines else '',encoding='utf-8')
spec = {'path':str(YOLO_DIR),'train':'images/train','val':'images/val',
        'names':{idx:CAT_NAMES[cid] for idx,cid in INDEX2CAT.items()}}
YOLO_YAML.write_text(yaml.safe_dump(spec,sort_keys=False,allow_unicode=True))
pd.DataFrame(conversion).to_csv(PROOF/'konversi_yolo.csv',index=False)
assert not (YOLO_DIR/'images'/'test').exists()
assert len(list((YOLO_DIR/'images'/'train').glob('*.jpg')))==len(COCO_DATA['train']['images'])
print('YOLO hanya berisi Train/Valid:',YOLO_YAML,'| test tidak disalin ke data training.')


## 7. Dataset Mask R-CNN dan model

Mask R-CNN membaca mask RLE canonical. Class output 1..K, background 0; YOLO memakai 0..K-1. FrozenBatchNorm dipertahankan saat training/load; evaluasi tidak mengunduh pretrained lagi. Resize internal `min_size=max_size=640` membatasi sisi panjang; tensor kemudian dipad ke kelipatan 32. YOLO memakai letterbox square. Perbedaan padding dan recipe antararsitektur dilaporkan.

In [ ]:
class CocoMaskDataset(Dataset):
    def __init__(self,split):
        if split not in ('train','valid'): raise ValueError('Loader training hanya boleh train/valid.')
        self.split = split;self.images = COCO_DATA[split]['images']
    def __len__(self):return len(self.images)
    def __getitem__(self,idx):
        info = self.images[idx];arr = load_rgb(self.split,info);h,w = arr.shape[:2]
        boxes,masks,labels,areas,crowds = [],[],[],[],[]
        for a in COCO_GT[self.split].imgToAnns[info['id']]:
            mask = decode_seg(a['segmentation'],h,w)
            if not mask.any():continue
            yy,xx = np.where(mask)
            boxes.append([int(xx.min()),int(yy.min()),int(xx.max())+1,int(yy.max())+1])
            masks.append(mask);labels.append(CAT2INDEX[a['category_id']]+1)
            areas.append(float(mask.sum()));crowds.append(int(a.get('iscrowd',0)))
        target = {'boxes':torch.tensor(boxes,dtype=torch.float32).reshape(-1,4),
            'labels':torch.tensor(labels,dtype=torch.int64),
            'masks':torch.from_numpy(np.stack(masks)) if masks else torch.zeros((0,h,w),dtype=torch.uint8),
            'image_id':torch.tensor(info['id'],dtype=torch.int64),
            'area':torch.tensor(areas,dtype=torch.float32),'iscrowd':torch.tensor(crowds,dtype=torch.int64)}
        return to_tensor(arr),target

def detection_collate(batch):return tuple(zip(*batch))
def build_maskrcnn(pretrained=False,size=BASE_SIZE):
    # Arsitektur normalisasi identik saat train dan load. Tidak mengunduh bobot saat evaluasi.
    from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
    fpn = resnet_fpn_backbone(backbone_name='resnet50',weights=None,
                             norm_layer=FrozenBatchNorm2d,trainable_layers=3)
    model = torchvision.models.detection.MaskRCNN(fpn,num_classes=91,
        min_size=size,max_size=size,box_detections_per_img=PRED_MAX_DET,
        box_score_thresh=PRED_CONF,box_nms_thresh=0.7)
    if pretrained:
        weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.DEFAULT
        state = weights.get_state_dict(progress=True,check_hash=True)
        model.load_state_dict(state,strict=True)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,NC+1)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,NC+1)
    return model

def save_mask_checkpoint(path,model,epoch,steps):
    temp = Path(str(path)+'.tmp')
    torch.save({'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                'fingerprint':DATA_FINGERPRINT,'category_ids':CAT_IDS,'size':BASE_SIZE,
                'epoch':int(epoch),'steps':int(steps)},temp)
    temp.replace(path)
def load_mask_model(path,size=None):
    # weights_only=True; checkpoint hanya tensor dan tipe Python dasar.
    ck = torch.load(path,map_location='cpu',weights_only=True)
    assert ck['fingerprint']==DATA_FINGERPRINT and ck['category_ids']==CAT_IDS,'Checkpoint berasal dari data/kelas berbeda.'
    model = build_maskrcnn(pretrained=False,size=size or ck['size'])
    model.load_state_dict(ck['state_dict'],strict=True)
    return model.to(DEVICE).eval()
print('Contoh target Mask R-CNN:',{k:tuple(v.shape) for k,v in CocoMaskDataset('train')[0][1].items()})


## 8. Inferensi dan evaluator mask yang sama

Semua prediksi dikembalikan ke dimensi GT. Inferensi membatasi sisi panjang sebelum jaringan, lalu meng-upsample setiap mask agar foto 4080 piksel tidak menghasilkan tensor mask GPU raksasa. Ini konsisten pada seluruh varian; detail interpolasi dicatat. Ambang AP=0.001; diagnostik kegagalan=0.5. Max detections=100 untuk semua model. Waktu inferensi meliputi resize, forward, postprocess/encoding dan transfer; tidak meliputi loading model atau disk decode; panggilan pertama dikecualikan dari rata-rata.

In [ ]:
def resize_longside(arr,size):
    h,w = arr.shape[:2];scale = size/max(h,w)
    nw,nh = max(1,round(w*scale)),max(1,round(h*scale))
    return cv2.resize(arr,(nw,nh),interpolation=cv2.INTER_LINEAR)

def model_predict_image(model,kind,arr,size):
    # Inferensi pada gambar berukuran terbatas, lalu setiap mask diproyeksi balik satu per satu.
    # Ini menghindari tensor 100 x 4080 x 2296 di GPU ketika foto HP beresolusi tinggi.
    small = resize_longside(arr,size);h,w = arr.shape[:2];result = []
    if kind=='yolo':
        res = model.predict(cv2.cvtColor(small,cv2.COLOR_RGB2BGR),imgsz=size,
            device=0 if DEVICE.type=='cuda' else 'cpu',conf=PRED_CONF,iou=0.7,
            max_det=PRED_MAX_DET,retina_masks=True,rect=False,verbose=False)[0]
        if res.masks is not None:
            for box,mask in zip(res.boxes,res.masks.data):
                idx = int(box.cls.item());score = float(box.conf.item())
                if idx not in INDEX2CAT: raise ValueError('Output kelas YOLO tidak cocok dengan mapping.')
                m = mask.detach().cpu().numpy()
                full = cv2.resize(m.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST)
                if full.any():result.append({'category_id':INDEX2CAT[idx],'segmentation':encode_mask(full),'score':score})
        del res
    else:
        with torch.inference_mode(): output = model([to_tensor(small).to(DEVICE)])[0]
        labels = output['labels'].cpu().tolist();scores = output['scores'].cpu().tolist()
        for k,(lab,score) in enumerate(zip(labels,scores)):
            if score<PRED_CONF:continue
            if lab-1 not in INDEX2CAT:raise ValueError('Output kelas Mask R-CNN tidak cocok.')
            m = output['masks'][k,0].detach().float().cpu().numpy()
            full = (cv2.resize(m,(w,h),interpolation=cv2.INTER_LINEAR)>=0.5).astype(np.uint8)
            if full.any():result.append({'category_id':INDEX2CAT[lab-1],'segmentation':encode_mask(full),'score':float(score)})
        del output
    return result

def infer_split(model,kind,split,image_ids,size,save_dir=None):
    ids = set(image_ids);selected = [i for i in COCO_DATA[split]['images'] if i['id'] in ids]
    if len(selected)!=len(ids) or not selected:raise ValueError('Subset evaluasi tidak valid/kosong')
    rows,timings = [],[]
    for k,info in enumerate(selected):
        arr = load_rgb(split,info)
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        start = time.perf_counter()
        pred = model_predict_image(model,kind,arr,size)
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        elapsed = time.perf_counter()-start
        for p in pred:p['image_id']=int(info['id'])
        rows.extend(pred)
        timings.append({'image_id':int(info['id']),'seconds':elapsed,'n_pred':len(pred),'first_call':k==0})
        if (k+1)%50==0:print('Inferensi',split,k+1,'/',len(selected))
    if save_dir:
        save_dir.mkdir(parents=True,exist_ok=True)
        write_json(save_dir/'predictions_coco.json',rows)
        pd.DataFrame(timings).to_csv(save_dir/'inference_timing.csv',index=False)
    return rows,timings

def coco_evaluate(split,preds,ids,cats,verbose=False):
    if not cats:raise ValueError('Scope kategori kosong')
    gt = COCO_GT[split]
    # bbox prediksi tidak dikirim: COCOeval segmentation menggunakan luas mask, bukan luas kotak.
    cleaned = [{'image_id':int(p['image_id']),'category_id':int(p['category_id']),
                'segmentation':copy.deepcopy(p['segmentation']),'score':float(p['score'])} for p in preds]
    with redirect_stdout(io.StringIO()):
        if cleaned:dt = gt.loadRes(cleaned)
        else:dt = make_coco({'info':{},'images':COCO_DATA[split]['images'],'categories':CATEGORIES,'annotations':[]})
        ev = COCOeval(gt,dt,'segm')
        ev.params.imgIds = sorted(ids);ev.params.catIds = sorted(cats)
        ev.params.maxDets = [1,10,100]
        ev.evaluate();ev.accumulate();ev.summarize()
    if verbose:print('AP50-95, AP50, AP75:',ev.stats[:3])
    return ev

def eval_metrics(ev):
    out = {}
    for key,index in [('mask_AP50_95',0),('mask_AP50',1),('mask_AP75',2),('mask_AR100',8)]:
        value = float(ev.stats[index]);out[key] = value if value>=0 else None
    return out

def eval_per_class(ev):
    rows=[]
    for k,c in enumerate(ev.params.catIds):
        p = ev.eval['precision'][:,:,k,0,-1];values = p[p>=0]
        rows.append({'category_id':c,'class':CAT_NAMES[c],'mask_AP50_95':float(values.mean()) if values.size else None,
                     'train_instances':train_freq[c],'gt_present':bool(values.size)})
    return rows

def load_run_model(kind,checkpoint,size):
    return YOLO(str(checkpoint)) if kind=='yolo' else load_mask_model(checkpoint,size)

def evaluate_checkpoint(kind,checkpoint,split,ids,run_name,size=BASE_SIZE,scopes=None):
    checkpoint = Path(checkpoint)
    if not checkpoint.is_file():raise FileNotFoundError(checkpoint)
    out = METRICS/run_name;out.mkdir(parents=True,exist_ok=True)
    key = {'dataset':DATA_FINGERPRINT,'checkpoint_sha256':sha_file(checkpoint),
           'image_ids':sorted(ids),'size':size,'conf':PRED_CONF,'max_det':PRED_MAX_DET,'preprocess':'longside-v1'}
    manifest = out/'prediction_manifest.json';pred_path = out/'predictions_coco.json'
    if manifest.is_file() and json.loads(manifest.read_text())==key and pred_path.is_file():
        preds = json.loads(pred_path.read_text());timings = pd.read_csv(out/'inference_timing.csv').to_dict('records')
    else:
        model = load_run_model(kind,checkpoint,size)
        try:preds,timings = infer_split(model,kind,split,ids,size,out)
        finally:
            del model;gc.collect()
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
        write_json(manifest,key)
    active_scopes = scopes or SCOPES
    rows=[]
    for scope,cats in active_scopes.items():
        if not cats:continue
        ev = coco_evaluate(split,preds,ids,cats)
        per = eval_per_class(ev);pd.DataFrame(per).to_csv(out/f'AP_per_class_{scope}.csv',index=False)
        warm = [x['seconds'] for x in timings if not x.get('first_call',False)]
        rows.append({'run':run_name,'model':kind,'split':split,'scope':scope,'n_images':len(ids),
            'n_categories_requested':len(cats),'n_categories_with_gt':sum(r['gt_present'] for r in per),
            'size':size,'checkpoint':str(checkpoint),'seconds_per_image_excl_first':float(np.mean(warm)) if warm else None,
            'hp_labels_reviewed':HP_LABELS_REVIEWED,'source_v5_confirmed':SOURCE_V5_CONFIRMED,**eval_metrics(ev)})
    pd.DataFrame(rows).to_csv(out/'metrics.csv',index=False)
    return rows


## 9. Pelatihan, checkpoint, dan log

Pembagian train/valid ditetapkan sebelum eksperimen. Checkpoint YOLO dipilih oleh validation fitness bawaan, Mask R-CNN oleh validation mask AP. Semua dievaluasi lagi dengan evaluator COCO mask yang sama. Model tidak dipilih dari test. Rerun melewati run selesai yang konfigurasinya sama; checkpoint/data berbeda tidak boleh tercampur. Run terputus tidak otomatis dianggap selesai. Log dan heartbeat membantu mencatat waktu yang benar-benar dipakai.

In [ ]:
def utc():return datetime.now(timezone.utc).isoformat(timespec='seconds')
def append_csv(path,row):
    new = not path.exists()
    with path.open('a',newline='',encoding='utf-8') as f:
        writer = csv.DictWriter(f,fieldnames=list(row))
        if new:writer.writeheader()
        writer.writerow(row);f.flush()
TIME_LOG = PROOF/'durasi_training.csv'
TASK_LOG = PROOF/'logbook.csv'
def log(stage,status,note):append_csv(TASK_LOG,{'utc':utc(),'stage':stage,'status':status,'note':note})
def used_seconds(model_name):
    total = 0.; logged = set()
    if TIME_LOG.exists():
        table = pd.read_csv(TIME_LOG)
        subset = table[table.model==model_name]
        total = float(subset.seconds.sum()); logged = set(subset.run)
    # Bila kernel mati tanpa finally, heartbeat masih dihitung sebagai batas bawah waktu terpakai.
    for path in RUNS.glob('*/run_manifest.json'):
        run = path.parent.name
        meta = json.loads(path.read_text())
        hb = path.parent/'heartbeat.json'
        if meta.get('budget_key')==model_name and run not in logged and hb.is_file():
            total += float(json.loads(hb.read_text()).get('elapsed_seconds',0.))
    return total
def training_guard(model_name,minutes):
    if not SOURCE_V5_CONFIRMED:
        print('Provenance v5 masih terbuka. Run ini menggunakan data publik pada ekspor v2 yang diaudit.')
    if DEVICE.type!='cuda':raise RuntimeError('Training utama memerlukan GPU Kaggle. Audit dapat berjalan di CPU.')
    remaining = MAX_SECONDS_PER_MODEL-used_seconds(model_name)-120
    if remaining<60:raise RuntimeError(f'Anggaran training {model_name} habis.')
    return min(float(minutes)*60,remaining)
def log_time(model_name,run_name,start,status):
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    elapsed = time.monotonic()-start
    append_csv(TIME_LOG,{'utc':utc(),'model':model_name,'run':run_name,'seconds':round(elapsed,3),
                         'status':status,'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU'})
    log('training',status,f'{run_name}: {elapsed:.1f} detik wall-clock GPU, termasuk validasi/checkpoint dalam training call')

def run_meta(kind,size,minutes):
    return {'dataset':DATA_FINGERPRINT,'category_ids':CAT_IDS,'kind':kind,'size':size,
            'minutes':minutes,'seed':SEED,'epochs':EPOCHS,'ultralytics':importlib.metadata.version('ultralytics'),
            'batch':BATCH_MASK if kind=='maskrcnn' else BATCH_YOLO,'policy':'cpmk3-v3-no-random-split'}
def completed_checkpoint(run_dir,expected,checkpoint_name):
    manifest = run_dir/'run_manifest.json';complete = run_dir/'complete.json';ckpt = run_dir/checkpoint_name
    if manifest.exists() and json.loads(manifest.read_text())!=expected:
        raise RuntimeError(f'Konfigurasi run berubah di {run_dir}; gunakan output baru, jangan menimpa checkpoint lama.')
    if complete.is_file() and ckpt.is_file():return ckpt
    if manifest.exists():
        raise RuntimeError(f'Run {run_dir.name} belum selesai. Simpan bukti kegagalan; gunakan run ID baru untuk mengulang dan hitung waktu yang sudah dipakai.')
    return None

def train_yolo(run_name,weights,minutes,size,budget_key):
    folder = RUNS/run_name;meta = run_meta(weights,size,minutes);meta['budget_key']=budget_key
    existing = completed_checkpoint(folder,meta,'weights/best.pt')
    if existing:return existing
    seconds = training_guard(budget_key,minutes)
    seed_all();model=YOLO(weights)  # Unduh bobot sebelum penghitungan training.
    folder.mkdir(parents=True,exist_ok=True);write_json(folder/'run_manifest.json',meta)
    started = time.monotonic();status='failed'
    heartbeat = folder/'heartbeat.json'
    def beat(trainer):
        elapsed=time.monotonic()-started
        write_json(heartbeat,{'elapsed_seconds':elapsed,'epoch':int(trainer.epoch)+1,'utc':utc()})
        if elapsed>=seconds:trainer.stop=True
    model.add_callback('on_fit_epoch_end',beat)
    try:
        model.train(data=str(YOLO_YAML),imgsz=size,epochs=EPOCHS,batch=BATCH_YOLO,device=0,
            workers=WORKERS,seed=SEED,deterministic=True,project=str(RUNS),name=run_name,
            exist_ok=True,time=max(0.01,seconds/3600),patience=0,save=True,plots=True,
            amp=True,cache=False,multi_scale=False,rect=False,optimizer='SGD',lr0=0.01,
            momentum=0.937,weight_decay=0.0005,close_mosaic=0,
            hsv_h=0.015,hsv_s=0.7,hsv_v=0.4,degrees=0.0,translate=0.1,scale=0.5,
            shear=0.0,perspective=0.0,flipud=0.0,fliplr=0.5,mosaic=1.0,mixup=0.0,copy_paste=0.0)
        checkpoint=folder/'weights/best.pt'
        if not checkpoint.exists():raise RuntimeError('YOLO belum menghasilkan best.pt. Periksa results.csv dan error.')
        status='complete';write_json(folder/'complete.json',{'utc':utc(),'checkpoint':str(checkpoint)})
        return checkpoint
    finally:
        log_time(budget_key,run_name,started,status)
        del model;gc.collect();torch.cuda.empty_cache()

def train_mask(run_name='maskrcnn_640',minutes=MASK_MINUTES):
    folder=RUNS/run_name;meta=run_meta('maskrcnn',BASE_SIZE,minutes);meta['budget_key']='MaskRCNN'
    existing=completed_checkpoint(folder,meta,'best.pth')
    if existing:return existing
    seconds=training_guard('MaskRCNN',minutes)
    seed_all();model=build_maskrcnn(pretrained=True).to(DEVICE)
    loader=DataLoader(CocoMaskDataset('train'),batch_size=BATCH_MASK,shuffle=True,
        num_workers=WORKERS,collate_fn=detection_collate,generator=torch.Generator().manual_seed(SEED))
    optimizer=torch.optim.SGD([p for p in model.parameters() if p.requires_grad],lr=MASK_LR,momentum=0.9,weight_decay=0.0005)
    folder.mkdir(parents=True,exist_ok=True);write_json(folder/'run_manifest.json',meta)
    started=time.monotonic();status='failed';best_ap=-1.;step=0
    try:
        stop=False
        for epoch in range(EPOCHS):
            model.train();losses=[]
            for images,targets in loader:
                if time.monotonic()-started>=seconds:stop=True;break
                images=[x.to(DEVICE) for x in images]
                targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
                # Warmup sederhana kemudian LR tetap; dicatat, tidak memilih LR memakai test.
                lr=MASK_LR*min(1.,(step+1)/100)
                for g in optimizer.param_groups:g['lr']=lr
                ld=model(images,targets);loss=sum(ld.values())
                if not torch.isfinite(loss):raise FloatingPointError('Loss Mask R-CNN nonfinite.')
                optimizer.zero_grad(set_to_none=True);loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),10.);optimizer.step();step+=1
                losses.append(float(loss.detach().cpu()))
                write_json(folder/'heartbeat.json',{'elapsed_seconds':time.monotonic()-started,'steps':step,'utc':utc()})
            if not losses:break
            save_mask_checkpoint(folder/'last.pth',model,epoch+1,step)
            # Validation GT asli/canonical, setiap epoch. Waktu validasi masuk anggaran.
            model.eval()
            predictions,_=infer_split(model,'maskrcnn','valid',{i['id'] for i in COCO_DATA['valid']['images']},BASE_SIZE)
            ev=coco_evaluate('valid',predictions,{i['id'] for i in COCO_DATA['valid']['images']},SEEN_CAT_IDS)
            ap=float(ev.stats[0])
            append_csv(folder/'maskrcnn_train.csv',{'epoch':epoch+1,'steps':step,'train_loss':float(np.mean(losses)),
                'valid_mask_AP50_95':ap,'elapsed_seconds':time.monotonic()-started,'lr':lr})
            if ap>best_ap:
                best_ap=ap;shutil.copy2(folder/'last.pth',folder/'best.pth')
            print(f'Mask R-CNN epoch {epoch+1}: loss={np.mean(losses):.4f}, validation AP={ap:.4f}')
            if stop or time.monotonic()-started>=seconds:break
        if not (folder/'best.pth').exists():raise RuntimeError('Belum ada checkpoint Mask R-CNN yang tervalidasi.')
        status='complete';write_json(folder/'complete.json',{'utc':utc(),'best_validation_AP':best_ap})
        return folder/'best.pth'
    finally:
        log_time('MaskRCNN',run_name,started,status)
        del model;gc.collect();torch.cuda.empty_cache()

write_json(PROOF/'environment.json',{'utc':utc(),'python':sys.version,'torch':torch.__version__,
    'torchvision':torchvision.__version__,'ultralytics':importlib.metadata.version('ultralytics'),
    'seed':SEED,'nrp_digit':NRP_LAST_DIGIT,'device':str(DEVICE),'source_v5_note':SOURCE_V5_NOTE,
    'fingerprint':DATA_FINGERPRINT,'training_enabled':RUN_TRAINING})
log('audit','completed','Data/geometry audited; semantik label HP masih membutuhkan review manusia.')


## 10. Jalankan baseline, NRP 0, dan evaluasi

Mode audit melewati training; tidak mengeluarkan metrik palsu. Mode training mengerjakan baseline dan dua resolusi tambahan. Varian 640 adalah checkpoint baseline yang sama. Pemilihan ukuran didasarkan validation, tidak mempengaruhi baseline 640 yang sudah ditetapkan. Test asli dan HP diproses terpisah; status review/provenance disimpan bersama setiap baris hasil.

In [ ]:
assert DEVICE.type == 'cuda', 'Pilih GPU untuk run tambahan.'
print('Tipe GPU dan jumlah perangkat:',torch.cuda.get_device_name(0),torch.cuda.device_count())
VAL_IDS = {i['id'] for i in COCO_DATA['valid']['images']}
completed = []
for size in (320, 960):
    name = f'yolo11s_{size}'
    print('Mulai pelatihan',name,flush=True)
    path = train_yolo(name,'yolo11s-seg.pt',YOLO_S_MINUTES,size,'YOLO11s')
    completed.append(name)
    evaluate_checkpoint('yolo',path,'valid',VAL_IDS,f'nrp_{name}_valid',size=size,
                        scopes={'kelas_ada_di_train':SEEN_CAT_IDS})
print('CPMK3_EXTRA_RUNS_COMPLETE', ' '.join(completed))
for name in completed:
    assert (RUNS/name/'complete.json').is_file()
display(pd.read_csv(TIME_LOG))
